<a href="https://colab.research.google.com/github/grehna29/AI-ML-Lab/blob/main/AI_ML_MOD_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.svm import SVR, SVC
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_squared_error, r2_score, accuracy_score



CSV_FILE = "datalab_export_2026-09-29 09_09_35.csv"

if not Path(CSV_FILE).exists():
    raise FileNotFoundError(
        f"CSV file not found: {CSV_FILE}\n"
        "Put the Python file and CSV file in the same folder."
    )

data = pd.read_csv(CSV_FILE)

print("=" * 60)
print("DATASET INFORMATION")
print("=" * 60)
print("Shape:", data.shape)
print("Columns:", len(data.columns))
print()


# ============================================================
# 2. PREPARE DATA
# ============================================================

# Remove the index column because it is only an identifier.
if "index" in data.columns:
    data = data.drop(columns=["index"])

# Remove columns having only one unique value.
# Such columns contain no useful information for prediction.
constant_columns = [
    col for col in data.columns
    if data[col].nunique(dropna=False) <= 1
]

if constant_columns:
    print("Removed constant columns:", constant_columns)
    data = data.drop(columns=constant_columns)

# Fill any missing numeric values with the median.
numeric_columns = data.select_dtypes(include="number").columns
data[numeric_columns] = data[numeric_columns].fillna(
    data[numeric_columns].median()
)


# ============================================================
# 3. REGRESSION DATA
# ============================================================
# Regression target:
#     avg_daily_rate
#
# This is a continuous numerical variable in your CSV.

REGRESSION_TARGET = "avg_daily_rate"

if REGRESSION_TARGET not in data.columns:
    raise ValueError(
        f"Regression target '{REGRESSION_TARGET}' was not found in the CSV."
    )

# Do not use the classification target as a predictor for regression.
X_reg = data.drop(columns=[REGRESSION_TARGET, "is_canceled"], errors="ignore")
y_reg = data[REGRESSION_TARGET]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)



DATASET INFORMATION
Shape: (100, 53)
Columns: 53

Removed constant columns: ['arrival_date_week_number', 'arrival_date_month', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'hotel_City', 'hotel_Resort', 'meal_No_meal', 'market_segment_Aviation', 'market_segment_Complementary', 'market_segment_Groups', 'market_segment_Undefined', 'distribution_channel_GDS', 'distribution_channel_Undefined', 'reserved_room_type_B', 'reserved_room_type_L', 'deposit_type_No_Deposit', 'deposit_type_Non_Refund', 'deposit_type_Refundable', 'customer_type_Group']


In [ ]:

# ============================================================
# 4. DECISION TREE REGRESSION
# ============================================================

print("=" * 60)
print("1. DECISION TREE REGRESSION")
print("=" * 60)

dt_reg = DecisionTreeRegressor(random_state=42)
dt_reg.fit(X_train_reg, y_train_reg)

y_pred_dt_reg = dt_reg.predict(X_test_reg)

print("Mean Squared Error:",
      mean_squared_error(y_test_reg, y_pred_dt_reg))
print("R2 Score:",
      r2_score(y_test_reg, y_pred_dt_reg))
print()


# ============================================================
# 5. SUPPORT VECTOR MACHINE REGRESSION
# ============================================================
# SVR is sensitive to feature scale, so StandardScaler is used.

print("=" * 60)
print("2. SUPPORT VECTOR MACHINE REGRESSION")
print("=" * 60)

scaler_reg = StandardScaler()

X_train_reg_scaled = scaler_reg.fit_transform(X_train_reg)
X_test_reg_scaled = scaler_reg.transform(X_test_reg)

svr = SVR()
svr.fit(X_train_reg_scaled, y_train_reg)

y_pred_svr = svr.predict(X_test_reg_scaled)

print("Mean Squared Error:",
      mean_squared_error(y_test_reg, y_pred_svr))
print("R2 Score:",
      r2_score(y_test_reg, y_pred_svr))
print()


# ============================================================
# 6. RANDOM FOREST REGRESSION
# ============================================================

print("=" * 60)
print("3. RANDOM FOREST REGRESSION")
print("=" * 60)

rf_reg = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_reg.fit(X_train_reg, y_train_reg)

y_pred_rf_reg = rf_reg.predict(X_test_reg)

print("Mean Squared Error:",
      mean_squared_error(y_test_reg, y_pred_rf_reg))
print("R2 Score:",
      r2_score(y_test_reg, y_pred_rf_reg))
print()



1. DECISION TREE REGRESSION
Mean Squared Error: 1095.0091
R2 Score: 0.07130469250064064

2. SUPPORT VECTOR MACHINE REGRESSION
Mean Squared Error: 1152.9868004778946
R2 Score: 0.022132846921070537

3. RANDOM FOREST REGRESSION
Mean Squared Error: 601.0189411475005
R2 Score: 0.4902659070487019



In [ ]:
# ============================================================
# 7. CLASSIFICATION DATA
# ============================================================
# Classification target:
#     is_canceled
#
# 0 = Not Canceled
# 1 = Canceled

CLASSIFICATION_TARGET = "is_canceled"

if CLASSIFICATION_TARGET not in data.columns:
    raise ValueError(
        f"Classification target '{CLASSIFICATION_TARGET}' "
        "was not found in the CSV."
    )

# Do not use avg_daily_rate as a predictor here because it is the
# regression target used above.
X_cls = data.drop(
    columns=[CLASSIFICATION_TARGET, REGRESSION_TARGET],
    errors="ignore"
)
y_cls = data[CLASSIFICATION_TARGET]

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)


# ============================================================
# 8. DECISION TREE CLASSIFICATION
# ============================================================

print("=" * 60)
print("4. DECISION TREE CLASSIFIER")
print("=" * 60)

dt_cls = DecisionTreeClassifier(random_state=42)
dt_cls.fit(X_train_cls, y_train_cls)

y_pred_dt_cls = dt_cls.predict(X_test_cls)

print("Accuracy:",
      accuracy_score(y_test_cls, y_pred_dt_cls))
print()


# ============================================================
# 9. SUPPORT VECTOR MACHINE CLASSIFICATION
# ============================================================
# SVC is sensitive to feature scale, so StandardScaler is used.

print("=" * 60)
print("5. SUPPORT VECTOR MACHINE CLASSIFIER")
print("=" * 60)

scaler_cls = StandardScaler()

X_train_cls_scaled = scaler_cls.fit_transform(X_train_cls)
X_test_cls_scaled = scaler_cls.transform(X_test_cls)

svc = SVC()
svc.fit(X_train_cls_scaled, y_train_cls)

y_pred_svc = svc.predict(X_test_cls_scaled)

print("Accuracy:",
      accuracy_score(y_test_cls, y_pred_svc))
print()


# ============================================================
# 10. RANDOM FOREST CLASSIFICATION
# ============================================================

print("=" * 60)
print("6. RANDOM FOREST CLASSIFIER")
print("=" * 60)

rf_cls = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_cls.fit(X_train_cls, y_train_cls)

y_pred_rf_cls = rf_cls.predict(X_test_cls)

print("Accuracy:",
      accuracy_score(y_test_cls, y_pred_rf_cls))
print()

4. DECISION TREE CLASSIFIER
Accuracy: 0.8

5. SUPPORT VECTOR MACHINE CLASSIFIER
Accuracy: 0.75

6. RANDOM FOREST CLASSIFIER
Accuracy: 0.9



In [ ]:


# ============================================================
# 11. SUMMARY
# ============================================================

print("=" * 60)
print("SUMMARY")
print("=" * 60)

print("REGRESSION")
print("- Decision Tree Regression")
print("  MSE:", mean_squared_error(y_test_reg, y_pred_dt_reg))
print("  R2 :", r2_score(y_test_reg, y_pred_dt_reg))

print("- Support Vector Machine Regression")
print("  MSE:", mean_squared_error(y_test_reg, y_pred_svr))
print("  R2 :", r2_score(y_test_reg, y_pred_svr))

print("- Random Forest Regression")
print("  MSE:", mean_squared_error(y_test_reg, y_pred_rf_reg))
print("  R2 :", r2_score(y_test_reg, y_pred_rf_reg))

print()
print("CLASSIFICATION")
print("- Decision Tree Classifier")
print("  Accuracy:", accuracy_score(y_test_cls, y_pred_dt_cls)*100)

print("- Support Vector Machine Classifier")
print("  Accuracy:", accuracy_score(y_test_cls, y_pred_svc)*100)

print("- Random Forest Classifier")
print("  Accuracy:", accuracy_score(y_test_cls, y_pred_rf_cls)*100)

print()
print("All six models completed successfully.")

SUMMARY
REGRESSION
- Decision Tree Regression
  MSE: 1095.0091
  R2 : 0.07130469250064064
- Support Vector Machine Regression
  MSE: 1152.9868004778946
  R2 : 0.022132846921070537
- Random Forest Regression
  MSE: 601.0189411475005
  R2 : 0.4902659070487019

CLASSIFICATION
- Decision Tree Classifier
  Accuracy: 80.0
- Support Vector Machine Classifier
  Accuracy: 75.0
- Random Forest Classifier
  Accuracy: 90.0

All six models completed successfully.
